# CUA models on Colab (T4) — Fara1.5-4B / OpenWebRL-4B

Cheap GPU path for OM2W bakeoff. Weights are on GCS; this notebook copies them, installs vLLM, and smoke-tests inference.

**Runtime:** GPU (T4 is enough for 4B bf16 with `max-model-len=16384`).

In [ ]:
# Colab: Runtime → Change runtime type → T4 GPU
!nvidia-smi

In [ ]:
from pathlib import Path

MODELS = Path("/content/usersim-models")
MODELS.mkdir(parents=True, exist_ok=True)

GCS_BUCKET = "gs://ai-studio-bucket-347838016394-us-east1/usersim-models"
# from google.colab import auth; auth.authenticate_user()  # if gsutil needs auth

!gsutil -m cp -r {GCS_BUCKET}/Fara1.5-4B {MODELS}/
!gsutil -m cp -r {GCS_BUCKET}/OpenWebRL-4B {MODELS}/
print("done", list(MODELS.iterdir()))

In [ ]:
!pip install -q "vllm>=0.19.1" requests pillow

In [ ]:
import subprocess, time, requests

MODEL = str(MODELS / "Fara1.5-4B")
proc = subprocess.Popen([
    "vllm", "serve", MODEL,
    "--host", "0.0.0.0", "--port", "8000",
    "--dtype", "bfloat16",
    "--max-model-len", "16384",
    "--gpu-memory-utilization", "0.92",
    "--limit-mm-per-prompt", "image=5",
    "--trust-remote-code",
])
for _ in range(120):
    try:
        requests.get("http://127.0.0.1:8000/v1/models", timeout=2).raise_for_status()
        print("vLLM ready")
        break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not start")

In [ ]:
import json, requests
model_id = requests.get("http://127.0.0.1:8000/v1/models").json()["data"][0]["id"]
r = requests.post("http://127.0.0.1:8000/v1/chat/completions", json={
    "model": model_id,
    "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
    "max_tokens": 16,
    "temperature": 0,
}, timeout=120)
print(json.dumps(r.json(), indent=2))